In [3]:
# imports
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as plt
import tensorflow as tf
import random

ImportError: DLL load failed while importing _pywrap_custom_casts: The specified procedure could not be found.

In [2]:
import tensorflow as tf

print("TF Version:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

TF Version: 2.10.0
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
class_df = pd.read_csv("E:/Datasets/AIA-image-classification/images.csv")
folder = "E:/Datasets/AIA-image-classification/images"
class_df.head(5)

,image_name,label
0,0.jpg,0
1,1.jpg,4
2,2.jpg,5
3,4.jpg,0
4,7.jpg,4


In [4]:
clean_names = []
clean_paths = []
clean_labels = []
missing_files = []

for index, row in class_df.iterrows():
    image_id = str(row['image_name'])
    label = row['label']
    
    full_path = os.path.join(folder, image_id)
    
    if os.path.exists(full_path):
        clean_names.append(image_id)
        clean_paths.append(full_path)
        clean_labels.append(label)
    else:
        missing_files.append(image_id)

# ✅ Create dataframe (ALL columns aligned)
df = pd.DataFrame({
    'image_name': clean_names,
    'image_paths': clean_paths,
    'label': clean_labels
})

# Required for generator
df['label'] = df['label'].astype(str)

print("Missing files:", missing_files)
print(df.info())
print(df['label'].value_counts())
df.head(10)

Missing files: []
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14034 entries, 0 to 14033
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   image_name   14034 non-null  object
 1   image_paths  14034 non-null  object
 2   label        14034 non-null  object
dtypes: object(3)
memory usage: 329.0+ KB
None
label
3    2512
2    2404
5    2382
4    2274
1    2271
0    2191
Name: count, dtype: int64


,image_name,image_paths,label
0,0.jpg,E:/Datasets/AIA-image-classification/images\0.jpg,0
1,1.jpg,E:/Datasets/AIA-image-classification/images\1.jpg,4
2,2.jpg,E:/Datasets/AIA-image-classification/images\2.jpg,5
3,4.jpg,E:/Datasets/AIA-image-classification/images\4.jpg,0
4,7.jpg,E:/Datasets/AIA-image-classification/images\7.jpg,4
5,8.jpg,E:/Datasets/AIA-image-classification/images\8.jpg,1
6,9.jpg,E:/Datasets/AIA-image-classification/images\9.jpg,5
7,10.jpg,E:/Datasets/AIA-image-classification/images\10...,2
8,12.jpg,E:/Datasets/AIA-image-classification/images\12...,5
9,13.jpg,E:/Datasets/AIA-image-classification/images\13...,2


In [5]:
from PIL import Image

bad_images = []
for image in df['image_paths']:
    try:
        img = Image.open(image)  
        img.verify()             
    except:
        bad_images.append(image)

len(bad_images)

0

In [10]:
train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(128, 128),   # 🔥 smaller = faster + easier
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_iterator = val_generator.flow_from_dataframe(
    val_df,
    x_col='image_paths',
    y_col='label',
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 11227 validated image filenames belonging to 6 classes.
Found 2807 validated image filenames belonging to 6 classes.


In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    
    # Conv Block 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),
    
    # Conv Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    
    # Conv Block 3
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    # Conv Block 3
    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),


    layers.GlobalAveragePooling2D(),
    
    # Output
    layers.Dense(6, activation='softmax')
])

In [12]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_iterator,
    validation_data=val_iterator,
    epochs=5
)

Epoch 1/5
351/351 [==============================] - 26s 72ms/step - loss: 1.4010 - accuracy: 0.4148 - val_loss: 1.2342 - val_accuracy: 0.4845
Epoch 2/5
351/351 [==============================] - 26s 75ms/step - loss: 1.1705 - accuracy: 0.5278 - val_loss: 1.1812 - val_accuracy: 0.5561
Epoch 3/5
351/351 [==============================] - 27s 77ms/step - loss: 1.0901 - accuracy: 0.5685 - val_loss: 1.2171 - val_accuracy: 0.5422
Epoch 4/5
318/351 [==========================>...] - ETA: 2s - loss: 1.0583 - accuracy: 0.5914

In [6]:
# input split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_generator = ImageDataGenerator(
    rescale=1./255
)

In [9]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model
base_model.trainable = False

ImportError: `load_weights` requires h5py package when loading weights from HDF5. Try installing h5py.

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(6, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    save_best_only=True
)

In [ ]:
history = model.fit(
    train_iterator,
    validation_data=val_iterator,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

In [ ]:
loss, acc = model.evaluate(val_iterator)

print("🔥 Validation Accuracy:", acc)

In [ ]:
train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(224, 224),   # EfficientNet default
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

val_iterator = val_generator.flow_from_dataframe(
    val_df,
    x_col='image_paths',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [10]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.metrics import Precision, Recall, Accuracy

def create_model():
    model = Sequential([
        Conv2D(16, (3, 3), activation='relu', input_shape=(256, 256, 3)),
        MaxPooling2D(),
        BatchNormalization(),
        
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(),
        BatchNormalization(),
    
        GlobalAveragePooling2D(),
        Dense(128, activation='relu'),
        layers.Dropout(0.5),
        Dense(6, activation='softmax')    
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', Precision(), Recall()])
    return model

model = create_model()
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 254, 254, 16)      448       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 127, 127, 16)     0         
 )                                                               
                                                                 
 batch_normalization (BatchN  (None, 127, 127, 16)     64        
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 125, 125, 32)      4640      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 62, 62, 32)       0         
 2D)                                                             
                                                        

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    save_best_only=True
)

In [12]:
classes = sorted(df['label'].unique())
print("Classes:", classes)

Classes: ['0', '1', '2', '3', '4', '5']


In [14]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = []
precision_scores = []
recall_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(df)):
    
    print("Train:", train_iterator.class_indices)
    print("Val:", val_iterator.class_indices)
    print(f"\n🔥 Fold {fold+1}")

    # Split dataframe
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    # Generators
    train_iterator = train_generator.flow_from_dataframe(
    train_df,
    x_col='image_paths',
    y_col='label',
    target_size=(256, 256),
    batch_size=256,
    class_mode='categorical',
    classes=classes
    )

    val_iterator = val_generator.flow_from_dataframe(
        val_df,
        x_col='image_paths',
        y_col='label',
        target_size=(256, 256),
        batch_size=256,
        class_mode='categorical',
        classes=classes
    )
    
    model = create_model()

    # Train
    model.fit(
        train_iterator,
        validation_data=val_iterator,
        epochs=20,
        callbacks=[early_stop])

    # Evaluate
    loss, acc, precision, recall = model.evaluate(val_iterator)
    print("Accuracy:", acc)
    # Store scores
    acc_scores.append(acc)
    precision_scores.append(precision)
    recall_scores.append(recall)
print('🔥 Done !')

Train: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
Val: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}

🔥 Fold 1
Found 11227 validated image filenames belonging to 6 classes.
Found 2807 validated image filenames belonging to 6 classes.
Epoch 1/20
44/44 [==============================] - 77s 2s/step - loss: 1.1968 - accuracy: 0.5293 - precision_2: 0.6693 - recall_2: 0.3271 - val_loss: 2.0378 - val_accuracy: 0.1706 - val_precision_2: 0.0000e+00 - val_recall_2: 0.0000e+00
Epoch 2/20
23/44 [==============>...............] - ETA: 35s - loss: 1.0007 - accuracy: 0.6172 - precision_2: 0.7338 - recall_2: 0.4546

KeyboardInterrupt: 

In [ ]:
# Final result
print("🔥 Average Accuracy:", sum(acc_scores)/len(acc_scores))
print("🔥 Precision:", precision_scores)
print("🔥 Recall:", recall_scores)